In [73]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [31]:
d = pd.read_csv('train.csv')
d.sample()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
561,562,0,3,"Sivic, Mr. Husein",male,40.0,0,0,349251,7.8958,NaN,S


In [32]:
d = d.drop( columns=['Cabin','PassengerId','Ticket','Name'])

In [33]:
d.sample()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
309,1,1,female,30.0,0,0,56.9292,C


# train_test_split

In [42]:
x , x_test , y , y_test = train_test_split( d.drop( columns=['Survived']) , d['Survived'] , test_size=0.20 , random_state=2 )
#print( x.sample())
#print( y.sample() )

# imputation

In [26]:
d.isnull().sum()  # Age & Embarked 

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [56]:
a = SimpleImputer()
e = SimpleImputer( strategy = 'most_frequent' )

x[['Age']] =  (a.fit_transform( x[['Age']])).astype('int')
x_test[['Age']] = (a.fit_transform( x_test[['Age']])).astype('int')

x[['Embarked']] = e.fit_transform( x[['Embarked']])
x_test[['Embarked']] = e.fit_transform( x_test[['Embarked']])

# Encoding

In [59]:
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(x[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(x[['Embarked']])

X_test_sex = ohe_sex.transform(x_test[['Sex']])
X_test_embarked = ohe_embarked.transform(x_test[['Embarked']])

In [62]:
X_train_embarked

array([[1., 0., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       ...,
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 0., 1.]])

In [64]:
x_train_rem = x.drop(columns=['Sex','Embarked'])
x_test_rem = x_test.drop(columns=['Sex','Embarked'])

In [67]:
X_train_transformed = np.concatenate((x_train_rem,X_train_sex,X_train_embarked),axis=1)
X_test_transformed = np.concatenate((x_test_rem,X_test_sex,X_test_embarked),axis=1)

In [69]:
X_train_transformed.shape

(712, 10)

In [68]:
X_test_transformed.shape

(179, 10)

# Descision tree

In [70]:
t = DecisionTreeClassifier()

In [71]:
t.fit(X_train_transformed,y)

DecisionTreeClassifier()

In [74]:
y_p = t.predict( X_test_transformed )

In [75]:
accuracy_score(y_test,y_p)

0.770949720670391

# model

In [79]:
import os
# Create the directory if it doesn't exist
os.makedirs('models', exist_ok=True)

In [80]:
import pickle

# Save one-hot encoders
pickle.dump(ohe_sex, open('models/ohe_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('models/ohe_embarked.pkl', 'wb'))

# Save trained classifier model
pickle.dump(t, open('models/clf.pkl', 'wb'))
